# Goal statement:
Does Class II/III latency show the same sharp 2013→2019 decline-then-plateau shape, or is the plateau unique to Class I

## Subgoal:
Did FDA optimize the Class I pipeline specifically, leaving lower-severity recalls untouched

# Environment Setup & Storage Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Libraries Installation & Imports

In [17]:
!pip install pandera -q

import glob  # used to merge multiple CSV files from one folder in Google Drive
import os
import numpy as np
import altair as alt
import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Column, Check
import statsmodels.formula.api as smf  # used for the OLS backlog-latency regression

# Data Processing

## Lean Six Sigma Palette Configuration

Every chart below draws from one shared palette so the same color always carries the same meaning:


*   Blue — neutral process data / central tendency (no judgment implied)
*   Grey — contextual / denominator data (e.g., raw volume) — background information, not a signal
* Amber — warning zone / approaching an out-of-spec condition
* Red — out-of-spec / critical — reserved strictly for confirmed threshold breaches
* Dark neutral — statistical overlays (trend/fit lines). A model artifact, not a verdict

In [3]:
SIXSIGMA_PALETTE = {
    'blue':   '#1F6FB2',  # neutral process data / central tendency
    'grey':   '#8C8C8C',  # contextual / denominator data (no signal)
    'amber':  '#FFC000',  # warning zone
    'red':    '#C00000',  # out-of-spec / critical
    'dark':   '#404040',  # statistical overlay (trend/fit lines)
    'green':  '#548235',  # in-control / meets target (reserved, unused for now)
}

## File Path Matching & CSV Ingestion

In [4]:
# Define folder path
folder_path = (
    "/content/drive/MyDrive/FDA recall data analysis/Class 2 3 Latency/Raw data"
)
# Get the CSV files in the folder
all_files = glob.glob(f"{folder_path}/*.csv")
print(f"Loaded {len(all_files)} files.")

# Read CSV files and assign source file name as the trailing column (used for traceability)
df_list = [
    pd.read_csv(file, low_memory=False).assign(source_file=os.path.basename(file))
    for file in all_files
]

Loaded 22 files.


## Pre-Merge Schema Validation Audit

In [5]:
# Normalize column names across all DataFrames: strip whitespace, lowercase
for df in df_list:
    df.columns = df.columns.str.strip().str.lower()

# Build a per-file schema map and compare each file against the union of all columns
schema_map = {
    os.path.basename(f): set(df.columns) for f, df in zip(all_files, df_list)
}
all_columns = set.union(*schema_map.values())
common_columns = set.intersection(*schema_map.values())
missing_anywhere = all_columns - common_columns

print(f"Total Unique Columns Across Files: {len(all_columns)}")
print(f"Columns Common to ALL Files: {len(common_columns)}")
print(f"Columns Missing in at Least One File: {len(missing_anywhere)}\n")

print("-" * 60)
print("SCHEMA DIVERGENCE BY FILE")
print("-" * 60)

for file_name, cols in schema_map.items():
    missing = all_columns - cols
    extra = cols - common_columns
    if missing or extra:
        print(f"\n--- File: {file_name} ---")
        if missing:
            print(f"  Missing columns ({len(missing)}): {sorted(list(missing))}")
        if extra:
            print(f"  Extra columns ({len(extra)}): {sorted(list(extra))}")

Total Unique Columns Across Files: 29
Columns Common to ALL Files: 26
Columns Missing in at Least One File: 3

------------------------------------------------------------
SCHEMA DIVERGENCE BY FILE
------------------------------------------------------------

--- File: Class 1 Drug recalls Jan 2024 to 12 July 2026.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Class 1 Drug recalls Jan 2013 to Dec 2018.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Class 1 Drug recalls Jan 2019 to Dec 2023.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: 08 Jun 2012 to Dec 2012.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Jan 2013 to Jun 2013.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Jul 2013 to Dec 2013.csv ---
  Missing columns (3)

## Code Info Field Reconstruction

Applies to Jan 2016 to Oct 2016.csv only. A field-length overflow during export split Code Info across three extra columns for 2 rows (Event ID 74057, Event ID 74745). Reconstructs the full string and drops the overflow columns from that file's DataFrame before Step 7's concat, so every file has a uniform column set going into the merge.

In [6]:
# Target the single file known to carry the Code Info overflow
target_file = "Jan 2016 to Oct 2016.csv"
overflow_cols = ["more code info", "more code info.1", "more code info.2"]
affected_event_ids = [74057, 74745]

for f, df in zip(all_files, df_list):
    if os.path.basename(f) == target_file:
        present_overflow = [c for c in overflow_cols if c in df.columns]
        if not present_overflow:
            print(f"No overflow columns found in {target_file}; skipping reconstruction.")
            break

        mask = df["event id"].isin(affected_event_ids)
        print(f"Rows flagged for Code Info reconstruction in {target_file}: {mask.sum()}")

        # Rebuild the full Code Info string only for the affected rows
        pieces = ["code info"] + present_overflow
        df.loc[mask, "code info"] = (
            df.loc[mask, pieces]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
            .str.strip()
        )

        # Drop the overflow columns so this file matches the schema of every other file
        df.drop(columns=present_overflow, inplace=True)
        break

Rows flagged for Code Info reconstruction in Jan 2016 to Oct 2016.csv: 77


## Concatenation & Null Standardization Audit

Runs on the schema-uniform df_list produced by Step 6. Merges all files into one master structure and runs an initial missing-value audit. Only true blanks/whitespace are converted to NaN here. The placeholder tokens (e.g. "unknown", "n/a") are deliberately left untouched until Step 8's audited pass.

In [7]:
# Unify all files into a single master structure
combined_df = pd.concat(df_list, ignore_index=True)

# Convert empty strings / whitespace-only cells to true NaN
df_clean = combined_df.replace(r"^\s*$", np.nan, regex=True)

# Initial missing value audit on the unified record set
missing_summary = pd.DataFrame(
    {
        "Data Type": df_clean.dtypes,
        "Total Rows": len(df_clean),
        "Missing Count": df_clean.isna().sum(),
        "Missing (%)": (df_clean.isna().mean() * 100).round(2),
    }
)

print("=== INITIAL MISSING DATA AUDIT ===")
print(
    missing_summary[missing_summary["Missing Count"] > 0].sort_values(
        by="Missing Count", ascending=False
    )
)

=== INITIAL MISSING DATA AUDIT ===
                                                 Data Type  Total Rows  \
last modified date                                  object       17726   
address2                                            object       17726   
termination date                                    object       17726   
product quantity                                    object       17726   
state/province                                      object       17726   
initial firm notification of consignee or public    object       17726   
voluntary/mandated                                  object       17726   
code info                                           object       17726   
recall number                                       object       17726   
city                                                object       17726   
address1                                            object       17726   
distribution pattern                                object       17726   
cou

## Placeholder Null Discovery & Standardization

Scans every column for case-insensitive, whitespace-stripped matches against a placeholder token list discovered from an audit of the actual raw values. Flag in a new boolean column "was_reported_unknown" before conversion so an FDA-reported "unknown" stays distinguishable from a genuinely blank field. Then all matched values are converted to true NaN in a single uniform pass.

In [8]:
# Audit the most frequent normalized string values to identify placeholder tokens
# actually present in the data (rather than assuming a fixed list)
object_cols = df_clean.select_dtypes(include="object").columns
normalized_values = (
    df_clean[object_cols]
    .apply(lambda s: s.astype(str).str.strip().str.lower())
)
value_counts_sample = normalized_values.stack().value_counts()
print("Most frequent normalized string values (inspect for placeholder tokens):")
print(value_counts_sample.head(30))

# Placeholder tokens confirmed present in the audit above
placeholder_tokens = {
    "unknown", "none", "n/a", "na", "null", "nan",
    "not reported", "not applicable", "not recorded",
}

# Flag rows where ANY column carried a placeholder token, before converting anything
was_reported_unknown = pd.DataFrame(False, index=df_clean.index, columns=object_cols)
for col in object_cols:
    was_reported_unknown[col] = normalized_values[col].isin(placeholder_tokens)

df_clean["was_reported_unknown"] = was_reported_unknown.any(axis=1)

# Convert all matched placeholder values to true NaN in one uniform pass
for col in object_cols:
    df_clean.loc[normalized_values[col].isin(placeholder_tokens), col] = np.nan

n_flagged = df_clean["was_reported_unknown"].sum()
print("\nRows with at least one FDA-reported placeholder value:", n_flagged)

Most frequent normalized string values (inspect for placeholder tokens):
nan                                                                                  38983
drugs                                                                                17726
no                                                                                   17718
voluntary: firm initiated                                                            17662
united states                                                                        16689
terminated                                                                           14685
class ii                                                                             14336
letter                                                                                9057
nationwide                                                                            3152
telephone                                                                             2700
ongoing          

## Multi-Type Data Imputation & Cleanup

In [9]:
print("-" * 60)
# 1. APPLY CATEGORICAL / TEXT IMPUTATIONS
print("-" * 60)
# For text/metadata columns (e.g., lot numbers, firm names, reason for recall),
# fill missing values with explicit labels rather than dropping rows.
text_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c != "was_reported_unknown"]

for col in text_cols:
    if "lot" in col.lower():
        # Specific flag for lot number columns missing in newer files
        df_clean[col] = df_clean[col].fillna("Not Recorded")
    else:
        # General placeholder for other missing metadata text
        df_clean[col] = df_clean[col].fillna("Unknown")

print("-" * 60)
# 2. APPLY DATE / TEMPORAL IMPUTATIONS
print("-" * 60)
date_cols = [c for c in df_clean.columns if "date" in c.lower() or "time" in c.lower()]

for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

print("-" * 60)
# 3. APPLY NUMERIC IMPUTATIONS
print("-" * 60)
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if df_clean[col].isna().any():
        median_val = df_clean[col].median()
        n_filled = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"  {col}: filled {n_filled} missing values with median {median_val}")

print("-" * 60)
# 4. VERIFY REMAINING NULLS
print("-" * 60)
remaining_nulls = df_clean.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

if remaining_nulls.empty:
    print("No remaining nulls across text, date, and numeric fields.")
else:
    print("Remaining nulls by column:")
    print(remaining_nulls.sort_values(ascending=False))

------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
Remaining nulls by column:
recently updated record    17726
voluntary/mandated         17726
last modified date         16394
termination date            3038
dtype: int64


/tmp/ipykernel_10564/3246897849.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
/tmp/ipykernel_10564/3246897849.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
/tmp/ipykernel_10564/3246897849.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")


## Pre-Dedup Date Consistency Validation

Runs on the fully cleaned, date-parsed dataset. Confirms it is safe to collapse to one row per Event ID for latency calculations. Guards against silently computing latency from the wrong row if a future data pull violates the event-level date consistency observed here.

In [10]:
date_consistency = (
    df_clean.groupby('event id')[['recall initiation date', 'center classification date']]
    .nunique(dropna=True)
)

inconsistent_events = date_consistency[
    (date_consistency['recall initiation date'] > 1) |
    (date_consistency['center classification date'] > 1)
]

print(f"Event IDs with inconsistent core dates: {len(inconsistent_events)} of {date_consistency.shape[0]}")
if len(inconsistent_events) > 0:
    print(inconsistent_events)

Event IDs with inconsistent core dates: 0 of 4563


## Event Population Definition & Cross-Project Overlap Disclosure

Defines the Class I/II/III population as drop_duplicates(subset=['Event ID'], keep='first') applied to the classification-filtered slice. Same as the Class I project's dedup logic. No attempt to assign a single "resolved" classification per event. The known overlap with the Class I population is documented as an explicit, expected condition rather than a defect, since no FDA documentation defines a canonical event-level classification for events spanning multiple severities.

In [11]:
# Classification-filtered slice: Class I, II, or III events (all three now in scope)
class123_slice = df_clean[df_clean['classification'].isin(['Class I', 'Class II', 'Class III'])]

# Dedup logic mirrors the Class I project exactly: keep first occurrence per Event ID,
# with no attempt to assign a single "resolved" classification
class123_events = class123_slice.drop_duplicates(subset=['event id'], keep='first')

print(f"Class I/II/III event population: {len(class123_events)} unique Event IDs")
print(class123_events['classification'].value_counts())

# Cross-project reconciliation
class1_event_ids_path = (
    "/content/drive/MyDrive/FDA recall data analysis/Class 1 Latency/class1_event_ids.csv"
)
class1_events_here = class123_events.loc[
    class123_events['classification'] == 'Class I', 'event id'
]
print(f"\nClass I events found in the unified population: {len(class1_events_here)}")

try:
    class1_event_ids = pd.read_csv(class1_event_ids_path, usecols=['event id'])['event id']
    matched = class1_events_here.isin(class1_event_ids).sum()
    unmatched = len(class1_events_here) - matched
    print(f"Of these, also present in the separate Class I project's output: {matched}")
    if unmatched:
        print(
            f"Not found in the separate Class I project's output: {unmatched} -- "
            "review for classification or dedup discrepancies between the two projects."
        )
except FileNotFoundError:
    print(
        "Class I event ID reference not found at the expected path -- reconciliation "
        "skipped. This does not affect the population above, since Class I events are "
        "sourced directly from this dataset rather than from that external file."
    )

Class I/II/III event population: 4563 unique Event IDs
classification
Class II     2821
Class III    1118
Class I       624
Name: count, dtype: int64

Class I events found in the unified population: 624
Class I event ID reference not found at the expected path -- reconciliation skipped. This does not affect the population above, since Class I events are sourced directly from this dataset rather than from that external file.


# Data Analysis

## Core Latency Field & Year Assignment

In [12]:
# Compute core latency field
class123_events['Latency_Days'] = (
    class123_events['center classification date'] - class123_events['recall initiation date']
).dt.days

# Assign each event to a calendar year
class123_events['Year'] = class123_events['recall initiation date'].dt.year

class123_events['Report_Date'] = pd.to_datetime(class123_events['report date'], errors='coerce')
class123_events['Report_Lag'] = (
    class123_events['Report_Date'] - class123_events['recall initiation date']
).dt.days

PULL_DATE = pd.Timestamp('2026-07-12')  # TODO: confirm this matches the Class II/III raw data pull date
safety_margin = int(class123_events['Report_Lag'].quantile(0.90))
class123_events['Reliable'] = class123_events['recall initiation date'] <= (
    PULL_DATE - pd.Timedelta(days=safety_margin)
)

/tmp/ipykernel_10564/977343168.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  class123_events['Latency_Days'] = (
/tmp/ipykernel_10564/977343168.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  class123_events['Year'] = class123_events['recall initiation date'].dt.year
/tmp/ipykernel_10564/977343168.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

## Calculate backlog-signal metrics per year

* `total_recall_volume` runs on the full dataset — it counts recall
*initiations*, which are fully known the moment a recall starts, so there's
no censoring to correct for.

* `median_latency`, `p90_latency`, `pct_over_60`, and `pct_over_90`. All four are derived from `Latency_Days`, so a
recent year's resolved cases are a biased sample (skewed toward the fast
outcomes) unless still-too-recent events are excluded.

In [13]:
def pct_over(series, threshold):
    return (series.dropna() > threshold).mean() * 100

# Full-dataset metric -- total_recall_volume counts every initiated recall,
# regardless of resolution status, so it isn't subject to right-censoring
full_metrics = class123_events.groupby(['Year', 'classification']).agg(
    total_recall_volume=('event id', 'nunique'),
).reset_index()

# Reliable-only metrics -- median, P90, and threshold % are all latency-derived,
# so recent, still-open events would skew all four toward "faster" outcomes
reliable_events = class123_events[class123_events['Reliable']]
reliable_metrics = reliable_events.groupby(['Year', 'classification']).agg(
    median_latency=('Latency_Days', 'median'),
    p90_latency=('Latency_Days', lambda x: np.percentile(x.dropna(), 90)),
    pct_over_60=('Latency_Days', lambda x: pct_over(x, 60)),
    pct_over_90=('Latency_Days', lambda x: pct_over(x, 90)),
).reset_index()

annual_metrics = full_metrics.merge(reliable_metrics, on=['Year', 'classification'], how='left')

print("-" * 60)
print("Annual Backlog Metrics by Classification")
print("-" * 60)
print(annual_metrics.to_string(index=False))

------------------------------------------------------------
Annual Backlog Metrics by Classification
------------------------------------------------------------
 Year classification  total_recall_volume  median_latency  p90_latency  pct_over_60  pct_over_90
 2007       Class II                    1          1901.0       1901.0   100.000000   100.000000
 2010       Class II                    2           952.5       1072.9   100.000000   100.000000
 2011        Class I                    2           758.0        975.6   100.000000   100.000000
 2011       Class II                   13           345.0        621.4   100.000000   100.000000
 2011      Class III                    6           616.5        764.0   100.000000   100.000000
 2012        Class I                   19           150.0        371.2    84.210526    68.421053
 2012       Class II                  118            72.5        153.9    56.779661    33.898305
 2012      Class III                   60            57.0    

## Volume-Backlog Relationship Test (event-level)

Runs on `class123_events` after Step 12. Computes `Concurrent_Volume` per
event -- an O(n log n) rolling count, via `searchsorted`, of all recalls
initiated within a +/-45-day window -- then tests whether latency scales with
concurrent volume via raw correlation, an OLS regression controlling for
year and classification, and a group-mean-detrended cross-check.

**Adapted from the Class I script, two changes:**
- `Concurrent_Volume` intentionally counts recalls system-wide (any
  classification), not same-class only -- it's meant to capture FDA-wide
  review-queue pressure, which plausibly affects latency for any class
  waiting in that queue, so this was left as-is.
- `C(classification)` was added to the regression and to the detrending
  group keys. The Class I-only script had no classification variable to
  control for; now that the population spans three classes with likely
  different baseline latency, omitting it would let between-class latency
  differences masquerade as a volume effect. `import statsmodels.formula.api
  as smf` was added to Step 2's imports to support this.

In [14]:
# Concurrent_Volume via searchsorted -- O(n log n)
WINDOW_DAYS = 45
dates = class123_events['recall initiation date'].values
order = np.argsort(dates)
sorted_dates = dates[order]
left = np.searchsorted(sorted_dates, sorted_dates - np.timedelta64(WINDOW_DAYS, 'D'), side='left')
right = np.searchsorted(sorted_dates, sorted_dates + np.timedelta64(WINDOW_DAYS, 'D'), side='right')
counts = np.empty(len(dates), dtype=int)
counts[order] = right - left - 1  # exclude self
class123_events['Concurrent_Volume'] = counts
reg_df = class123_events.dropna(subset=['Latency_Days', 'Concurrent_Volume', 'Year', 'classification']).copy()

# Raw correlation + regression (coef, p-value, CI need statsmodels either way)
raw_corr = reg_df['Latency_Days'].corr(reg_df['Concurrent_Volume'])
full_model = smf.ols('Latency_Days ~ Concurrent_Volume + C(Year) + C(classification)', data=reg_df).fit()
coef, pval = full_model.params['Concurrent_Volume'], full_model.pvalues['Concurrent_Volume']
ci_low, ci_high = full_model.conf_int().loc['Concurrent_Volume']

# Detrend both variables via group-mean subtraction (by Year and classification)
reg_df['Detrended_Latency'] = reg_df['Latency_Days'] - reg_df.groupby(['Year', 'classification'])['Latency_Days'].transform('mean')
reg_df['Detrended_Volume'] = reg_df['Concurrent_Volume'] - reg_df.groupby(['Year', 'classification'])['Concurrent_Volume'].transform('mean')

print(f"Raw correlation: {raw_corr:.4f}")
print(f"Regression coef (controlling for Year, Classification): {coef:.4f}, p={pval:.4f}, 95% CI [{ci_low:.4f}, {ci_high:.4f}]")

Raw correlation: -0.2057
Regression coef (controlling for Year, Classification): -0.0651, p=0.4891, 95% CI [-0.2497, 0.1194]


/tmp/ipykernel_10564/61783552.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  class123_events['Concurrent_Volume'] = counts


## Queue-Depth (Open-Caseload) Time Series

Runs on `class123_events` after Step 12. For each month-end snapshot date `T`,
counts events still open at that moment (`recall initiation date <= T` and
`center classification date > T`), computed **per classification** rather
than pooled -- so Class I, II, and III backlog trajectories can be plotted
and compared on the same time axis in the visualizations below.

**Adapted from the Class I script:** the original computed one open-count per
snapshot across a single class. Grouping by `classification` inside each
snapshot is the only structural change; the snapshot loop and event-open
condition are otherwise unchanged. This step exists purely to produce
`queue_df` for the Monthly Queue Depth chart -- it wasn't shown in the
screenshots you pasted earlier, but the chart in Step 18 depends on it.

In [15]:
# For each month-end date T in the data range, count events where:
# recall initiation date <= T and center classification date > T -- computed
# per classification so Class I/II/III backlog trajectories are comparable

start = class123_events['recall initiation date'].min()
end = class123_events['recall initiation date'].max()
snapshots = pd.date_range(start=start, end=end, freq='ME')  # month-end dates

queue_depth = []
all_classes = class123_events['classification'].unique()
for snap in snapshots:
    open_events = class123_events[
        (class123_events['recall initiation date'] <= snap) &
        (class123_events['center classification date'] > snap)
    ]
    open_counts = open_events.groupby('classification')['event id'].nunique()
    for cls in all_classes:
        queue_depth.append({
            'Snapshot': snap,
            'classification': cls,
            'Open_Events': int(open_counts.get(cls, 0)),
        })

queue_df = pd.DataFrame(queue_depth)

print("-" * 60)
print("Queue Depth Time Series (tail, all classes)")
print("-" * 60)
print(queue_df.tail(12).to_string(index=False))

------------------------------------------------------------
Queue Depth Time Series (tail, all classes)
------------------------------------------------------------
  Snapshot classification  Open_Events
2026-03-31        Class I            2
2026-03-31       Class II           19
2026-03-31      Class III            2
2026-04-30        Class I            3
2026-04-30       Class II           19
2026-04-30      Class III            3
2026-05-31        Class I            2
2026-05-31       Class II           19
2026-05-31      Class III            1
2026-06-30        Class I            0
2026-06-30       Class II           12
2026-06-30      Class III            0


# Data Visualization

## Median vs P90 Latency by Year, by Classification

Runs on `annual_metrics` from Step 13. **Adapted from the Class I script:**
`melt`'s `id_vars` now includes `classification` alongside `Year`, and the
chart is row-faceted by `classification` (with a shared y-axis) instead of
a single panel -- a straight color-by-classification overlay on one panel
would have made three overlapping median/P90 line pairs hard to read
against each other. Faceting keeps the Median-vs-P90 color coding intact
per class while stacking the three classes for direct visual comparison,
which is the actual question this notebook is asking.

In [18]:
latency_long = annual_metrics.melt(
    id_vars=['Year', 'classification'],
    value_vars=['median_latency', 'p90_latency'],
    var_name='Metric',
    value_name='Days'
)
latency_long['Metric'] = latency_long['Metric'].map({
    'median_latency': 'Median',
    'p90_latency': '90th percentile'
})

latency_chart = alt.Chart(latency_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('Days:Q', title='Latency (days)'),
    color=alt.Color('Metric:N',
                     scale=alt.Scale(domain=['Median', '90th percentile'],
                                      range=[SIXSIGMA_PALETTE['blue'], SIXSIGMA_PALETTE['amber']]),
                     legend=alt.Legend(title=None, orient='top')),
    strokeDash=alt.StrokeDash('Metric:N', legend=None),
    tooltip=['Year:O', 'classification:N', 'Metric:N', alt.Tooltip('Days:Q', format='.1f')]
).properties(
    width=650, height=180
).facet(
    row=alt.Row('classification:N', title=None, header=alt.Header(labelFontWeight='bold', labelFontSize=13)),
    title='Median vs 90th Percentile Latency by Year, by Classification'
).resolve_scale(y='shared')

latency_chart

alt.FacetChart(...)

## Percentage of Recalls Exceeding Threshold, by Year and Classification

Runs on `annual_metrics`. **Adapted from the Class I script:** same
`id_vars`/faceting change as Step 16, for the same reason -- the underlying
`pct_over_60`/`pct_over_90` bar-pair encoding is otherwise untouched.

In [ ]:
pct_long = annual_metrics.melt(
    id_vars=['Year', 'classification'],
    value_vars=['pct_over_60', 'pct_over_90'],
    var_name='Threshold',
    value_name='Percent'
)
pct_long['Threshold'] = pct_long['Threshold'].map({
    'pct_over_60': '> 60 days',
    'pct_over_90': '> 90 days'
})

pct_chart = alt.Chart(pct_long).mark_bar().encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('Percent:Q', title='% of recalls exceeding threshold'),
    color=alt.Color('Threshold:N',
                     scale=alt.Scale(domain=['> 60 days', '> 90 days'],
                                      range=[SIXSIGMA_PALETTE['amber'], SIXSIGMA_PALETTE['red']]),
                     legend=alt.Legend(title=None, orient='top')),
    xOffset='Threshold:N',
    tooltip=['Year:O', 'classification:N', 'Threshold:N', alt.Tooltip('Percent:Q', format='.1f')]
).properties(
    width=650, height=180
).facet(
    row=alt.Row('classification:N', title=None, header=alt.Header(labelFontWeight='bold', labelFontSize=13)),
    title='Share of Recalls Exceeding Latency Threshold, by Year and Classification'
).resolve_scale(y='shared')

pct_chart

## FDA Recall Monthly Queue Depth by Classification (2013-2026)

Runs on `queue_df` from Step 15. **Adapted from the Class I script, one
substantive change:** the original rendered a single gradient-filled area
(`mark_area` with a linear gradient), which only reads cleanly for one
series. With three classes now in `queue_df`, three overlapping gradient
areas would occlude each other and misrepresent overlap as intensity. This
was switched to a `mark_line` chart colored by `classification` -- the
honest way to compare three time series on a shared axis without a
stacking or occlusion artifact. If you want the filled-area look back,
that would need faceting or `mark_area` with `opacity` low enough that
overlaps stay legible; say so and I'll swap it in.

In [ ]:
queue_chart = alt.Chart(queue_df).mark_line(strokeWidth=2, interpolate='monotone').encode(
    x=alt.X('Snapshot:T', title='Month', axis=alt.Axis(format='%Y')),
    y=alt.Y('Open_Events:Q', title='Open events (month-end)'),
    color=alt.Color('classification:N',
                     scale=alt.Scale(domain=['Class I', 'Class II', 'Class III'],
                                      range=[SIXSIGMA_PALETTE['blue'], SIXSIGMA_PALETTE['amber'], SIXSIGMA_PALETTE['grey']]),
                     legend=alt.Legend(title=None, orient='top')),
    tooltip=[alt.Tooltip('Snapshot:T', title='Month', format='%b %Y'),
             'classification:N',
             alt.Tooltip('Open_Events:Q', title='Open events')]
).properties(
    title='FDA Recall Monthly Queue Depth by Classification (2013-2026)',
    width=700, height=350
)

queue_chart

## Year-Level Volume by Classification

Runs on `annual_metrics`. **Adapted from the Class I script:** the original
was a single grey bar series (one class, so color carried no information).
With three classes, `color`/`xOffset` by `classification` is now needed just
to keep each year's three bars distinguishable -- otherwise `total_recall_volume`
across classes would stack or overwrite silently. Combined with `latency_chart`
below it via `&`, same as the original.

In [ ]:
volume_chart = alt.Chart(annual_metrics).mark_bar().encode(
    x=alt.X('Year:O', title=None),
    y=alt.Y('total_recall_volume:Q', title='Total recalls (volume)'),
    color=alt.Color('classification:N',
                     scale=alt.Scale(domain=['Class I', 'Class II', 'Class III'],
                                      range=[SIXSIGMA_PALETTE['blue'], SIXSIGMA_PALETTE['amber'], SIXSIGMA_PALETTE['grey']]),
                     legend=alt.Legend(title=None, orient='top')),
    xOffset='classification:N',
    tooltip=['Year:O', 'classification:N', 'total_recall_volume:Q']
).properties(width=650, height=180)

volume_chart & latency_chart

## Concurrent Volume vs Detrended Latency, by Classification

Runs on `reg_df` from Step 14. **Adapted from the Class I script, two
changes, both following directly from Step 14's design:**
- Points are colored by `classification`, since `Detrended_Latency` and
  `Detrended_Volume` were computed by subtracting `['Year', 'classification']`
  group means -- both the year effect and the between-class baseline
  latency difference are already removed, so a single-color scatter would
  hide which class each point belongs to for no reason.
- The trend line uses `transform_regression(..., groupby=['classification'])`
  to fit one line per class instead of one pooled line. A single pooled
  trend would answer "is there a volume-latency relationship overall,"
  which Step 14's regression coefficient already answers numerically; the
  per-class lines answer the sharper question of whether that relationship
  differs by class -- e.g., if backlog predicts latency for Class II/III
  but not Class I, that's a different finding than a uniform effect, and a
  single trend line would visually erase it.

In [ ]:
class_scale = alt.Scale(domain=['Class I', 'Class II', 'Class III'],
                         range=[SIXSIGMA_PALETTE['blue'], SIXSIGMA_PALETTE['amber'], SIXSIGMA_PALETTE['grey']])

points = alt.Chart(reg_df).mark_circle(size=40, opacity=0.45).encode(
    x=alt.X('Detrended_Volume:Q', title='Concurrent volume (year + classification effect removed)'),
    y=alt.Y('Detrended_Latency:Q', title='Latency (days, year + classification effect removed)'),
    color=alt.Color('classification:N', scale=class_scale, legend=alt.Legend(title=None, orient='top')),
    tooltip=[alt.Tooltip('Detrended_Volume:Q', format='.1f'),
             alt.Tooltip('Detrended_Latency:Q', format='.1f'), 'Year:O', 'classification:N']
)

trend = points.transform_regression(
    'Detrended_Volume', 'Detrended_Latency', groupby=['classification']
).mark_line(strokeWidth=2).encode(
    color=alt.Color('classification:N', scale=class_scale, legend=None)
)

(points + trend).properties(
    title='Concurrent Volume vs Detrended Latency, by Classification (Backlog-vs-Load Test)',
    width=650, height=350
)

# Image Explorting

## Export All Charts to Google Drive

**Adapted from the Class I script, one change:** output directory renamed to
`charts_class123` (was `charts`) so this run doesn't silently overwrite the
Class I project's saved PNGs in the shared Drive folder. Discovery logic
(scan `globals()` for any Altair chart object) is unchanged.

In [ ]:
# Set up directory -- separate from the Class I project's charts folder
output_dir = '/content/drive/MyDrive/FDA_Recall_Analysis/charts_class123'
os.makedirs(output_dir, exist_ok=True)

# Automatically find all charts
exported_count = 0
for var_name, var_value in list(globals().items()):
    # Check if the variable is an Altair Chart (and not a private variable)
    if isinstance(var_value, alt.TopLevelMixin) and not var_name.startswith('_'):
        file_path = os.path.join(output_dir, f"{var_name}.png")
        var_value.save(file_path)
        print(f"Saved: {var_name}.png")
        exported_count += 1

print(f"\nSuccessfully exported {exported_count} chart(s) to: {output_dir}")